**RAG Debugging**

In [26]:
import os
import time
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from huggingface_hub import get_collection
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google import genai
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq



In [27]:
parser = StrOutputParser()

In [28]:
#all api keys

#QDRANT
QDRANT_API_KEY = os.getenv("QDRANTAPIKEY")
QDRANT_ENDPOINT = os.getenv("QDRANTENDPOINT")

#gemini model apikey
geminiapikey = os.getenv("GEMINIAPIKEY")

#
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     google_api_key=geminiapikey,
#     temperature=0
# )


groq_api_key = os.getenv("GROQAPIKEY")  # make sure this matches your .env variable name

llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=groq_api_key,
    temperature=0
)

In [29]:
start = time.time()
try:
    test = llm.invoke("Say hello in one word.")
    print("SUCCESS:", test)
except Exception as e:
    print("FAILED:", e)
print(f"Took {time.time() - start:.1f}s")

SUCCESS: content='<think>\nOkay, the user wants me to say hello in one word. Let me think about the possible options. The most straightforward is "Hello" itself, but maybe they want a different approach. Words like "Hi" or "Hey" are shorter, but "Hello" is the most direct. Are there any other single words that convey greeting? Maybe "Greetings" but that\'s a bit longer. "Salutations" is another option but it\'s more formal. The user probably expects a simple and common greeting. Let me confirm if there\'s any cultural or contextual nuance I should consider. Since the user didn\'t specify any particular language or context, sticking with "Hello" is safest. I\'ll go with that.\n</think>\n\nHello' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 14, 'total_tokens': 165, 'completion_time': 0.338716453, 'completion_tokens_details': None, 'prompt_time': 0.000570708, 'prompt_tokens_details': None, 'queue_time': 0.150460688, 'total_time': 0.339

In [10]:
# Embeddings model
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3666.51it/s]


In [11]:
file_path = "yarvalley.txt"
loader = TextLoader(file_path)
text_file = loader.load()

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=150,
    separators=[
        "\n\n",   # paragraphs (highest priority)
        "\n",     # lines
        ". ",     # sentences
        " ",      # words
        ""        # fallback
    ]
)

split_document = splitter.split_documents(text_file)


In [13]:
client = QdrantClient(
       url=QDRANT_ENDPOINT,
       api_key=QDRANT_API_KEY

)

In [14]:
collections = client.get_collections().collections


#Any go through list and stop at true(meet your condition)
collection_exist = any(
    collection.name == "ragval"
    for collection in collections
)

if not collection_exist:
    vectorestore = QdrantVectorStore.from_documents(
        documents=split_document,
        api_key=QDRANT_API_KEY,
        url=QDRANT_ENDPOINT,
        embedding=embeddings,
        collection_name="ragval"
    )
    print("New Collection Created")

else:
    vectorestore = QdrantVectorStore.from_existing_collection(
        collection_name="ragval",
        api_key=QDRANT_API_KEY,
        url=QDRANT_ENDPOINT,
        embedding=embeddings
    )

    print("Collection already exists. Using existing collection.")



Collection already exists. Using existing collection.


In [15]:
retriever = vectorestore.as_retriever(search_kwargs={"k":5})

In [16]:
rag_prompt = ChatPromptTemplate.from_template(
    """
     Please answer the following questions,if it is in context otherwise answer "I don,t have enough
     information about this.

     context:
     {context}

     question:
     {question}
    """
)

In [17]:
#A function that convert the retrieved,chunks of data into a string and remove the metadata
def get_content(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [18]:
rag_chain = (
    {
        "context": retriever | RunnableLambda(get_content),
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser  )

In [19]:
question = "Where is mingora"
answer = rag_chain.invoke(question)
print(answer)

<think>
Okay, let's see. The user is asking "Where is Mingora?" I need to check the provided context to find the answer.

Looking at the context under the "Town of Mingora" section. It says, "Mingora is the center of economic activities and the only urban area of the valley. It is adjacent to Saidu Sharif." Also, the Swat Valley is mentioned as being north of Peshawar. So Mingora is in the Swat Valley, near Saidu Sharif. The context doesn't mention a specific country, but since Swat is a region in Pakistan, I can infer that Mingora is in Pakistan. The answer should include that it's in the Swat Valley, adjacent to Saidu Sharif, and part of the area known as the Switzerland of the East. I should make sure not to add any extra info not in the context. The user might also be interested in knowing it's in Pakistan, but the context doesn't explicitly state the country, so maybe just stick to the given info. The answer should be concise and based solely on the provided context.
</think>

Min

In [20]:
question = "What is web scraping"

print("Answer: ", end="", flush=True)

# Use .stream() instead of .invoke()
for chunk in rag_chain.stream(question):
    print(chunk, end="", flush=True)

Answer: <think>
Okay, the user is asking, "What is web scraping?" Let me check the context provided to see if there's any information related to this.

Looking through the context, it's all about the Swat Valley in Pakistan—museums, cities, cultural aspects, and travel info. There's mention of the Swat Museum, Mingora town, local products, languages spoken, and some travel tips. Nowhere in the context does it talk about web scraping, technology, or data extraction from websites. 

The user's question is a general one about web scraping, which isn't covered in the provided context. Since the instructions say to answer "I don't have enough information about this" if it's not in the context, I should respond accordingly. I need to make sure I don't use any external knowledge and strictly rely on the given context. Yep, no mention of web scraping here. So the correct answer is to state the lack of information.
</think>

I don't have enough information about this.

**RAG ASSESMENT**

In [21]:
test_questions = [
    "Where is Swat?",
    "Where is Malam Jabba in Swat?",
    "What is Swat famous for?",
    "How many tourist spots are there in Swat?",
]

ground_truths = [
    "Swat Valley is located in the Malakand Division of Khyber Pakhtunkhwa province of Pakistan, situated north of Peshawar between 34°40' to 35°N latitude and 72° to 74°6'E longitude.",

    "Malam Jabba is located about 44 km from Mingora in Swat Valley. It is a modern hill resort featuring snowy mountain peaks, green valleys, forests, a chairlift, and a ski resort restored by TCKP in 2015.",

    "Swat is famous for its scenic natural beauty earning it the title Switzerland of the East, Buddhist civilization remnants and Gandhara art, emerald mines near Mingora, Malam Jabba ski resort, and tourist spots like Kalam, Bahrain, Madyan, Marghuzar, and Miandam.",

    "The document does not provide a specific numerical count of tourist spots, but it explicitly names and describes several key destinations and valleys in Swat, including: Mingora, Saidu Sharif(including the Swat Museum, the Tomb of the Akhund of Swat, and the Butkara archeological sites), Malam Jabba Ski Resort, Marghuzar, Miandam, Madyan, Bishigram Valley, Bahrain, Kalam (including Ushu, Utrot, Gabral, and Mahodand Lake), and Mankial Valley.",
]

In [22]:
#one is retrival as well as llm generation where we check the llm response with contetext second is only #vectore store context ,to check that the retrived chunks are accurate or not


answers = []
contexts = []

#Here we get the generated response by llm (also use the retrieved chunks)

for question in test_questions:
    answer = rag_chain.invoke(question)
    answers.append(answer)

#Here we get only retrieved chunks from vector DB

    retrieved_docs = retriever.invoke(question)
    retrieved_docs_content = [doc.page_content for doc in retrieved_docs]
    contexts.append(retrieved_docs_content)

In [23]:
import time
import pandas as pd

from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.run_config import RunConfig


C:\Users\Personal\AppData\Local\Temp\ipykernel_4060\1759065719.py:7: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Personal\AppData\Local\Temp\ipykernel_4060\1759065719.py:7: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Personal\AppData\Local\Temp\ipykernel_4060\1759065719.py:7: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ra

In [24]:
data = [
    {
        "user_input"  : test_questions[0],
        "response"    : answers[0],
        "retrieved_contexts" : contexts[0],
        "reference"   : ground_truths[0],
    },
    {
        "user_input"  : test_questions[1],
        "response"    : answers[1],
        "retrieved_contexts" : contexts[1],
        "reference"   : ground_truths[1],
    },
    {
        "user_input"  : test_questions[2],
        "response"    : answers[2],
        "retrieved_contexts" : contexts[2],
        "reference"   : ground_truths[2],
    },
    {
        "user_input"  : test_questions[3],
        "response"    : answers[3],
        "retrieved_contexts" : contexts[3],
        "reference"   : ground_truths[3],
    },
]

dataset = EvaluationDataset.from_list(data)

In [25]:
import os
import asyncio
import re
import pandas as pd
from dotenv import load_dotenv

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

# ===========================================================
# STEP 1 — LLM setup
# ===========================================================
load_dotenv()
groq_api_key = os.getenv("GROQAPIKEY")

groq_client = AsyncOpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# Comprehensive list of currently supported Groq production models
GROQ_MODELS = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "qwen/qwen3-32b",
    "qwen/qwen3.6-27b",
    "meta-llama/llama-4-scout-17b-16e-instruct"
]

current_model_index = 0
current_model_name = GROQ_MODELS[current_model_index]
judge_llm = llm_factory(current_model_name, client=groq_client)

# ===========================================================
# STEP 2 — Local embeddings
# ===========================================================
judge_embeddings = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

# ===========================================================
# STEP 3 — Metrics (rebuilt dynamically if model switches)
# ===========================================================
def build_metrics(llm):
    return {
        "faithfulness": Faithfulness(llm=llm),
        "answer_relevancy": AnswerRelevancy(llm=llm, embeddings=judge_embeddings),
        "context_precision": ContextPrecision(llm=llm),
        "context_recall": ContextRecall(llm=llm),
    }

metrics = build_metrics(judge_llm)

# ===========================================================
# STEP 4 — Retry wrapper with TPD and Token Limit awareness
# ===========================================================
async def score_with_retry(metric_name, build_call_fn, max_attempts=6, base_wait=60, max_wait=180):
    global judge_llm, metrics, current_model_name, current_model_index

    attempt = 1
    while attempt <= max_attempts:
        try:
            coro = build_call_fn(metrics[metric_name])
            result = await coro
            return result, None
        except Exception as e:
            err_str = str(e)
            print(f"    ⚠ {metric_name} attempt {attempt}/{max_attempts} failed: {err_str[:200]}")

            # Check for BOTH daily limits AND max_token output limits
            limit_keywords = ["tokens per day", "tpd", "max_tokens", "length limit"]
            if any(kw in err_str.lower() for kw in limit_keywords):
                if current_model_index < len(GROQ_MODELS) - 1:
                    current_model_index += 1
                    current_model_name = GROQ_MODELS[current_model_index]
                    print(f"    🔄 Hard limit reached. Switching to next model: {current_model_name}...")
                    judge_llm = llm_factory(current_model_name, client=groq_client)
                    metrics = build_metrics(judge_llm)
                    attempt = 1  # Reset attempts for the newly selected model
                    continue     # Retry immediately on the new model
                else:
                    print(f"    ❌ Limit reached on all available models. Exhausted fallback chain.")
                    return None, "Exhausted all models"

            # If it's a generic transient error (like a network timeout), wait and retry
            wait_time = min(base_wait * attempt, max_wait)
            if attempt < max_attempts:
                print(f"    ⏳ Waiting {wait_time}s before retrying {metric_name}...")
                await asyncio.sleep(wait_time)
            else:
                print(f"    ✗ {metric_name} gave up after {max_attempts} attempts.")
                return None, err_str

            attempt += 1

    return None, "Exhausted all attempts"

# ===========================================================
# STEP 5 — One sample, one metric at a time, 3 min apart
# ===========================================================
async def evaluate_sample(sample, gap_between_metrics=180):
    row = {
        "user_input": sample["user_input"],
        "response": sample["response"],
        "retrieved_contexts": sample["retrieved_contexts"],
        "reference": sample["reference"],
    }

    print("  → faithfulness...")
    result, err = await score_with_retry(
        "faithfulness",
        lambda m: m.ascore(
            user_input=sample["user_input"],
            response=sample["response"],
            retrieved_contexts=sample["retrieved_contexts"]
        )
    )
    row["faithfulness"] = float(result.value) if result is not None else None
    print(f"    ✓ faithfulness = {row['faithfulness']} (model used: {current_model_name})")
    print(f"    Resting {gap_between_metrics}s...")
    await asyncio.sleep(gap_between_metrics)

    print("  → answer_relevancy...")
    result, err = await score_with_retry(
        "answer_relevancy",
        lambda m: m.ascore(
            user_input=sample["user_input"],
            response=sample["response"]
        )
    )
    row["answer_relevancy"] = float(result.value) if result is not None else None
    print(f"    ✓ answer_relevancy = {row['answer_relevancy']} (model used: {current_model_name})")
    print(f"    Resting {gap_between_metrics}s...")
    await asyncio.sleep(gap_between_metrics)

    print("  → context_precision...")
    result, err = await score_with_retry(
        "context_precision",
        lambda m: m.ascore(
            user_input=sample["user_input"],
            retrieved_contexts=sample["retrieved_contexts"],
            reference=sample["reference"]
        )
    )
    row["context_precision"] = float(result.value) if result is not None else None
    print(f"    ✓ context_precision = {row['context_precision']} (model used: {current_model_name})")
    print(f"    Resting {gap_between_metrics}s...")
    await asyncio.sleep(gap_between_metrics)

    print("  → context_recall...")
    result, err = await score_with_retry(
        "context_recall",
        lambda m: m.ascore(
            user_input=sample["user_input"],
            retrieved_contexts=sample["retrieved_contexts"],
            reference=sample["reference"]
        )
    )
    row["context_recall"] = float(result.value) if result is not None else None
    print(f"    ✓ context_recall = {row['context_recall']} (model used: {current_model_name})")

    return row

# ===========================================================
# STEP 6 — All samples, 8 min apart
# ===========================================================
async def evaluate_all(data_list, gap_between_metrics=180, gap_between_samples=480):
    all_rows = []
    total = len(data_list)

    for idx, sample in enumerate(data_list):
        print(f"\n{'='*60}")
        print(f"Sample {idx+1}/{total}: {sample['user_input']}")
        print(f"{'='*60}")

        row = await evaluate_sample(sample, gap_between_metrics=gap_between_metrics)
        all_rows.append(row)
        print(f"\n✓ Sample {idx+1} fully completed: {row}")

        if idx + 1 < total:
            print(f"\nCooling down {gap_between_samples}s ({gap_between_samples/60:.1f} min)...")
            await asyncio.sleep(gap_between_samples)

    return pd.DataFrame(all_rows)

# ===========================================================
# STEP 7 — Clean Dataset (Remove <think> blocks)
# ===========================================================
def clean_dataset(data_list):
    cleaned_data = []
    for sample in data_list:
        clean_sample = sample.copy()

        # This regex removes everything from <think> to </think> (including the tags)
        # re.DOTALL ensures it matches across multiple lines
        clean_response = re.sub(r'<think>.*?</think>\s*', '', clean_sample["response"], flags=re.DOTALL)

        clean_sample["response"] = clean_response.strip()
        cleaned_data.append(clean_sample)
    return cleaned_data

# ===========================================================
# STEP 8 — Run it
# ===========================================================
# Clean the data before evaluating
cleaned_data = clean_dataset(data)

final_results = await evaluate_all(
    cleaned_data,
    gap_between_metrics=180,
    gap_between_samples=480
)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(final_results)

nan_rows = final_results[final_results.isna().any(axis=1)]
if len(nan_rows) > 0:
    print(f"\n⚠ {len(nan_rows)} rows still have missing values")
    print(nan_rows)
else:
    print("\n✓ All metrics completed successfully — zero missing values.")

final_results.to_csv("ragas_results.csv", index=False)
print("\n✓ Saved results to ragas_results.csv")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3491.55it/s]



Sample 1/4: Where is Swat?
  → faithfulness...
    ✓ faithfulness = 0.8333333333333334 (model used: llama-3.1-8b-instant)
    Resting 180s...
  → answer_relevancy...
    ✓ answer_relevancy = 0.9999999999999996 (model used: llama-3.1-8b-instant)
    Resting 180s...
  → context_precision...
    ✓ context_precision = 0.6791666666496875 (model used: llama-3.1-8b-instant)
    Resting 180s...
  → context_recall...
    ✓ context_recall = 1.0 (model used: llama-3.1-8b-instant)

✓ Sample 1 fully completed: {'user_input': 'Where is Swat?', 'response': '<think>\nOkay, the user is asking "Where is Swat?" Let me check the provided context.\n\nLooking at the context, the first section is titled "Valleys of Northern Swat" and the travel guide mentions Swat Valley in Pakistan. The travel information says Swat is connected to cities like Islamabad, Rawalpindi, and Peshawar via road. Also, there\'s mention of Malam Jabba being 44 km from Mingora, which is in Swat. The context clearly states that Swat i

API call failed on attempt 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kr36r9pvefka2tqadcf9aq22` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98928, Requested 2257. Please try again in 17m3.839999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Max retries exceeded. Total attempts: 1, Last error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kr36r9pvefka2tqadcf9aq22` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98928, Requested 2257. Please try again in 17m3.839999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


    ⚠ context_precision attempt 1/6 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kr36r9pvefka2tqadcf9aq22` service tier `on_demand` on tokens per day (TPD): Limit
    🔄 Hard limit reached. Switching to next model: openai/gpt-oss-20b...
    ✓ context_precision = 0.0 (model used: openai/gpt-oss-20b)
    Resting 180s...
  → context_recall...
    ✓ context_recall = 0.5 (model used: openai/gpt-oss-20b)

✓ Sample 4 fully completed: {'user_input': 'How many tourist spots are there in Swat?', 'response': '<think>\nOkay, let\'s see. The user is asking how many tourist spots there are in Swat. I need to check the provided context to find the answer.\n\nLooking at the context, there\'s a section about the Valleys of Northern Swat that mentions Malam Jabba Ski Resort. Then under Major Cities and Key Destinations, it talks about Mingora and Saidu Sharif. The context also mentions emerald mines north of Mingora. \n\nWa